In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# -------------------------
# Spark Session (YARN + HDFS)
# -------------------------
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName('HealthcareExtractJob') \
    .master('yarn') \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .config("spark.hadoop.yarn.resourcemanager.address", "resourcemanager:8032") \
    .config("spark.hadoop.yarn.resourcemanager.scheduler.address", "resourcemanager:8030") \
    .config("spark.driver.host", "172.30.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "512m") \
    .config("spark.yarn.am.memory", "512m") \
    .getOrCreate()

print("Spark Connected Successfully (Extract Job)")

# -------------------------
# Schema
# -------------------------
schema = StructType([
    StructField("Age", DoubleType(), True),
    StructField("Gender", StringType(), True),
    StructField("Medical Condition", StringType(), True),
    StructField("Admission Type", StringType(), True),
    StructField("Billing Amount", DoubleType(), True),
    StructField("Hospital", StringType(), True),
    StructField("Test Results", StringType(), True),
    StructField("patient_id", StringType(), True),
    StructField("event_time", StringType(), True)
])

# -------------------------
# Input (Landing Zone - Local or Mounted)
# -------------------------
input_path ="file:///opt/airflow/data/raw_healthcare_pings/"

print(f"Reading raw data from: {input_path}")

try:
    # Read raw data
    raw_df = spark.read \
        .schema(schema) \
        .option("recursiveFileLookup", "true") \
        .json(input_path)

    record_count = raw_df.count()
    print(f"Extracted {record_count} records")

    if record_count > 0:

        # -------------------------
        # Output → HDFS Bronze Layer
        # -------------------------
        bronze_path = "hdfs://hadoop-namenode:9000/user/jovyan/bronze/healthcare/"

        print(f"Writing to Bronze Layer: {bronze_path}")

        raw_df.write \
            .mode("append") \
            .format("parquet") \
            .save(bronze_path)

        print("Extract Job Completed Successfully (Data landed in Bronze)")

    else:
        print("No data found in input path")

except Exception as e:
    print(f"Extract Job Failed: {e}")

finally:
    spark.stop()
    print("Spark Session Stopped")

Spark Connected Successfully (Extract Job)
Reading raw data from: file:///opt/airflow/data/raw_healthcare_pings/
Extracted 1000 records
Writing to Bronze Layer: hdfs://hadoop-namenode:9000/user/jovyan/bronze/healthcare/
Extract Job Completed Successfully (Data landed in Bronze)
Spark Session Stopped
